### Preparing dataset for Netflix Content Dashboard

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Load and inspect dataset

In [2]:
df = pd.read_csv('data/netflix_titles.csv')

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


- Checking unique values for some columns

In [5]:
# show type
df['type'].value_counts()

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

In [6]:
# country
df['country'].value_counts()

country
United States                                    2818
India                                             972
United Kingdom                                    419
Japan                                             245
South Korea                                       199
                                                 ... 
Russia, Spain                                       1
Croatia, Slovenia, Serbia, Montenegro               1
Japan, Canada                                       1
United States, France, South Korea, Indonesia       1
United Arab Emirates, Jordan                        1
Name: count, Length: 748, dtype: int64

In [7]:
# rating
df['rating'].value_counts()

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64

- In the rating column, 74 min, 84 min, 66 min are irrelevent data (noise)
- We shall inspect those rows

In [8]:
df[df['rating'].str.contains('min', na=False)] # na ignores value rows

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


- Let's replace them using **mode**

In [9]:
mask = df['rating'].str.contains('min', na=False)

df.loc[mask, 'rating'] = df['rating'].mode()

In [10]:
df['rating'].value_counts().tail()

rating
NR          80
G           41
TV-Y7-FV     6
NC-17        3
UR           3
Name: count, dtype: int64

In [11]:
# listed in
df['listed_in'].value_counts()

listed_in
Dramas, International Movies                        362
Documentaries                                       359
Stand-Up Comedy                                     334
Comedies, Dramas, International Movies              274
Dramas, Independent Movies, International Movies    252
                                                   ... 
Action & Adventure, Cult Movies                       1
Action & Adventure, Comedies, Music & Musicals        1
Classic Movies, Horror Movies, Thrillers              1
Children & Family Movies, Classic Movies, Dramas      1
Cult Movies, Dramas, Thrillers                        1
Name: count, Length: 514, dtype: int64

- This column is quite useful, so we will feature engineer this column

In [12]:
# Creates separate columns for each genre found in 'listed_in'
genre_features = df['listed_in'].str.get_dummies(sep=', ')
df = pd.concat([df.drop('listed_in', axis=1), genre_features], axis=1)

In [13]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'description',
       'Action & Adventure', 'Anime Features', 'Anime Series',
       'British TV Shows', 'Children & Family Movies', 'Classic & Cult TV',
       'Classic Movies', 'Comedies', 'Crime TV Shows', 'Cult Movies',
       'Documentaries', 'Docuseries', 'Dramas', 'Faith & Spirituality',
       'Horror Movies', 'Independent Movies', 'International Movies',
       'International TV Shows', 'Kids' TV', 'Korean TV Shows', 'LGBTQ Movies',
       'Movies', 'Music & Musicals', 'Reality TV', 'Romantic Movies',
       'Romantic TV Shows', 'Sci-Fi & Fantasy', 'Science & Nature TV',
       'Spanish-Language TV Shows', 'Sports Movies', 'Stand-Up Comedy',
       'Stand-Up Comedy & Talk Shows', 'TV Action & Adventure', 'TV Comedies',
       'TV Dramas', 'TV Horror', 'TV Mysteries', 'TV Sci-Fi & Fantasy',
       'TV Shows', 'TV Thrillers', 'Teen TV Shows', 'Thrillers'],

### Performing data cleaning

In [14]:
# Remove unnecessary columns
df = df.drop(columns=['show_id', 'cast', 'description'])

- Handling missing values

In [15]:
df.isnull().sum().sort_values(ascending=False)

director                        2634
country                          831
date_added                        10
rating                             7
duration                           3
type                               0
title                              0
release_year                       0
Action & Adventure                 0
Anime Features                     0
Anime Series                       0
British TV Shows                   0
Children & Family Movies           0
Classic & Cult TV                  0
Classic Movies                     0
Comedies                           0
Crime TV Shows                     0
Cult Movies                        0
Documentaries                      0
Docuseries                         0
Dramas                             0
Faith & Spirituality               0
Horror Movies                      0
Independent Movies                 0
International Movies               0
International TV Shows             0
Kids' TV                           0
K

In [16]:
# Fill missing directors and countries with 'Unknown'
df['director'] = df['director'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

In [17]:
df.isnull().sum().sort_values(ascending=False)

date_added                      10
rating                           7
duration                         3
type                             0
title                            0
director                         0
release_year                     0
country                          0
Action & Adventure               0
Anime Features                   0
Anime Series                     0
British TV Shows                 0
Children & Family Movies         0
Classic & Cult TV                0
Classic Movies                   0
Comedies                         0
Crime TV Shows                   0
Cult Movies                      0
Documentaries                    0
Docuseries                       0
Dramas                           0
Faith & Spirituality             0
Horror Movies                    0
Independent Movies               0
International Movies             0
International TV Shows           0
Kids' TV                         0
Korean TV Shows                  0
LGBTQ Movies        

In [18]:
# Remove missing 'rating', 'date added' and 'duration' rows
df = df.dropna(subset=['rating', 'date_added', 'duration'])

----


In [19]:
df.head()

,type,title,director,country,date_added,release_year,rating,duration,Action & Adventure,Anime Features,...,TV Action & Adventure,TV Comedies,TV Dramas,TV Horror,TV Mysteries,TV Sci-Fi & Fantasy,TV Shows,TV Thrillers,Teen TV Shows,Thrillers
0,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,"September 25, 2021",2020,PG-13,90 min,0,0,...,0,0,0,0,0,0,0,0,0,0
1,TV Show,Blood & Water,Unknown,South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,0,0,...,0,0,1,0,1,0,0,0,0,0
2,TV Show,Ganglands,Julien Leclercq,Unknown,"September 24, 2021",2021,TV-MA,1 Season,0,0,...,1,0,0,0,0,0,0,0,0,0
3,TV Show,Jailbirds New Orleans,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,0,0,...,0,0,0,0,0,0,0,0,0,0
4,TV Show,Kota Factory,Unknown,India,"September 24, 2021",2021,TV-MA,2 Seasons,0,0,...,0,1,0,0,0,0,0,0,0,0
